In [7]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
c:\Users\Joaquin\Documents\GitHub\skforecast
0.23.0


In [8]:
# Libraries
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from skforecast.datasets import fetch_dataset
from skforecast.recursive import ForecasterRecursive
from skforecast.recursive import ForecasterRecursiveMultiSeries
from skforecast.utils import save_forecaster
from skforecast.utils import load_forecaster
from skforecast.preprocessing import CalendarFeatures
import joblib
import zipfile
import os

# Download data
# ==============================================================================
data = fetch_dataset(
    name="h2o", raw=True, kwargs_read={"names": ["y", "date"], "header": 0}, verbose=False
)
data['date'] = pd.to_datetime(data['date'], format='%Y-%m-%d')
data = data.set_index('date')
data = data.asfreq('MS')
data

,y
date,
1991-07-01,0.429795
1991-08-01,0.400906
1991-09-01,0.432159
1991-10-01,0.492543
1991-11-01,0.502369
...,...
2008-02-01,0.761822
2008-03-01,0.649435
2008-04-01,0.827887


In [9]:
# Fit export process
# ======================================================================================
calendar_transformer = CalendarFeatures(
    features = ['month', 'day_of_month'],
    keep_original_columns = True
)

data = calendar_transformer.fit_transform(X=data)
data_train = data.iloc[:190, :]
data_test = data.iloc[190:, :]


def exponential_weights(index):
    weights = np.exp(-np.arange(len(index)) * 0.01)[::-1]
    return weights / weights.sum() * len(index)

forecaster = ForecasterRecursive(
                 estimator     = RandomForestRegressor(random_state=123, max_depth=3),
                 lags          = 5,
                 forecaster_id = "forecaster_model",
                 transformer_y=StandardScaler(),
                 weight_func=exponential_weights  # custom function
             )

forecaster.fit(y=data_train['y'], exog=data_train.drop(columns='y'))
forecaster.predict(steps=1, exog=data_test.drop(columns='y'))

2007-05-01    0.666284
Freq: MS, Name: pred, dtype: float64

# Pickle

In [10]:
save_forecaster(forecaster, "forecaster.pkl", verbose= True, backend='pickle')

╭───────────────────────────── SaveLoadSkforecastWarning ──────────────────────────────╮
│ Custom function(s) used to create weights are defined in the '__main__' namespace    │
│ and have been saved as: 'exponential_weights.py'. These files must be imported       │
│ before loading the forecaster.                                                       │
│ Visit the documentation for more information:                                        │
│ https://skforecast.org/latest/user_guides/save-load-forecaster.html#saving-and-loadi │
│ ng-a-forecaster-model-with-custom-features                                           │
│                                                                                      │
│ Category : skforecast.exceptions.SaveLoadSkforecastWarning                           │
│ Location :                                                                           │
│ c:\Users\Joaquin\Documents\GitHub\skforecast\skforecast\utils\utils.py:2817          │
│ Suppress : warnings.simplefilter('ignore', category=SaveLoadSkforecastWarning)       │
╰──────────────────────────────────────────────────────────────────────────────────────╯

ForecasterRecursive 
Estimator: RandomForestRegressor 
Lags: [1 2 3 4 5] 
Window features: None 
Calendar features: None 
Window size: 5 
Series name: y 
Exogenous included: True 
Exogenous names: month_sin, month_cos, day_of_month_sin, day_of_month_cos 
Categorical features: auto 
Transformer for y: StandardScaler() 
Transformer for exog: None 
Weight function included: True 
Differentiation order: None 
Drop NaN from series: False 
Training range: [Timestamp('1991-07-01 00:00:00'), Timestamp('2007-04-01 00:00:00')] 
Training index type: DatetimeIndex 
Training index frequency: <MonthBegin> 
Estimator parameters: 
    {'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth':
    3, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None,
    'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2,
    'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100,
    'n_jobs': None, 'oob_score': False, 'random_state': 1

In [11]:
forecaster = load_forecaster("forecaster.pkl")
forecaster.predict(steps=1, exog=data_test.drop(columns='y'))

ForecasterRecursive 
Estimator: RandomForestRegressor 
Lags: [1 2 3 4 5] 
Window features: None 
Calendar features: None 
Window size: 5 
Series name: y 
Exogenous included: True 
Exogenous names: month_sin, month_cos, day_of_month_sin, day_of_month_cos 
Categorical features: auto 
Transformer for y: StandardScaler() 
Transformer for exog: None 
Weight function included: True 
Differentiation order: None 
Drop NaN from series: False 
Training range: [Timestamp('1991-07-01 00:00:00'), Timestamp('2007-04-01 00:00:00')] 
Training index type: DatetimeIndex 
Training index frequency: <MonthBegin> 
Estimator parameters: 
    {'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth':
    3, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None,
    'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2,
    'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100,
    'n_jobs': None, 'oob_score': False, 'random_state': 1

2007-05-01    0.666284
Freq: MS, Name: pred, dtype: float64

# Cloudpickle

In [12]:
save_forecaster(forecaster, "forecaster.cloudpickle", verbose= False, backend='cloudpickle')

In [13]:
forecaster = load_forecaster("forecaster.cloudpickle")
forecaster.predict(steps=1, exog=data_test.drop(columns='y'))

ForecasterRecursive 
Estimator: RandomForestRegressor 
Lags: [1 2 3 4 5] 
Window features: None 
Calendar features: None 
Window size: 5 
Series name: y 
Exogenous included: True 
Exogenous names: month_sin, month_cos, day_of_month_sin, day_of_month_cos 
Categorical features: auto 
Transformer for y: StandardScaler() 
Transformer for exog: None 
Weight function included: True 
Differentiation order: None 
Drop NaN from series: False 
Training range: [Timestamp('1991-07-01 00:00:00'), Timestamp('2007-04-01 00:00:00')] 
Training index type: DatetimeIndex 
Training index frequency: <MonthBegin> 
Estimator parameters: 
    {'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth':
    3, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None,
    'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2,
    'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100,
    'n_jobs': None, 'oob_score': False, 'random_state': 1

2007-05-01    0.666284
Freq: MS, Name: pred, dtype: float64

# Skops

In [14]:
save_forecaster(forecaster, "forecaster.skops", verbose= False, backend='skops')

╭───────────────────────────── SaveLoadSkforecastWarning ──────────────────────────────╮
│ Custom function(s) used to create weights are defined in the '__main__' namespace    │
│ and have been saved as: 'exponential_weights.py'. These files must be imported       │
│ before loading the forecaster.                                                       │
│ Visit the documentation for more information:                                        │
│ https://skforecast.org/latest/user_guides/save-load-forecaster.html#saving-and-loadi │
│ ng-a-forecaster-model-with-custom-features                                           │
│                                                                                      │
│ Category : skforecast.exceptions.SaveLoadSkforecastWarning                           │
│ Location :                                                                           │
│ c:\Users\Joaquin\Documents\GitHub\skforecast\skforecast\utils\utils.py:2817          │
│ Suppress : warnings.simplefilter('ignore', category=SaveLoadSkforecastWarning)       │
╰──────────────────────────────────────────────────────────────────────────────────────╯

In [15]:
forecaster = load_forecaster("forecaster.skops")
forecaster.predict(steps=1, exog=data_test.drop(columns='y'))

UntrustedTypesFoundException: Untrusted types found in the file: ['__main__.exponential_weights', 'numpy.dtype', 'pandas._libs.tslibs.offsets.MonthBegin', 'pandas.core.frame.DataFrame', 'pandas.core.indexes.datetimes.DatetimeIndex', 'skforecast.preprocessing._preprocessing.QuantileBinner', 'skforecast.recursive._forecaster_recursive.ForecasterRecursive'].
skops does not load these types unless you explicitly trust them. To review the full list of untrusted types in the file, run `skops.io.get_untrusted_types(file='forecaster.skops')`. If you trust the source of 'forecaster.skops', reload with `load_forecaster(..., trusted=True)` to trust them all, or pass the reviewed list via `trusted=[...]`.

In [ ]:
# Decompose last_window_ and training_range in values and index
last_window_dict = forecaster.last_window_.to_dict(orient="split")
last_window_dict["index"] = [str(ts) for ts in forecaster.last_window_.index]
last_window_dict['freq'] = forecaster.last_window_.index.freqstr
training_range_list = [str(ts) for ts in forecaster.training_range_]
forecaster.last_window_ = last_window_dict
forecaster.training_range_ = training_range_list

# Save object
dump(forecaster, "forecaster.skops")

# Load object
unknown_types = get_untrusted_types(file="forecaster.skops")
forecaster = load(file="forecaster.skops", trusted=unknown_types)

# Recostruct last_window_ and training_range
last_window_ = pd.DataFrame(
    data=forecaster.last_window_["data"], 
    index=pd.to_datetime(forecaster.last_window_["index"]),
    columns=forecaster.last_window_["columns"]
)
last_window_ = last_window_.asfreq(forecaster.last_window_["freq"])

training_range_ = pd.DatetimeIndex(forecaster.training_range_)
forecaster.last_window_ = last_window_
forecaster.training_range_ = training_range_

forecaster.predict(steps=1, exog=data_test)

In [ ]:
# Check what attributes can be serialized with skops
# ==============================================================================
import skops.io

# Assuming 'forecaster' is your trained skforecast object
print("Testing forecaster attributes for skops compatibility...\n")
print("-" * 50)

for attr_name, attr_value in vars(forecaster).items():
    try:
        # Attempt to serialize the individual attribute
        dumps(attr_value)
        print(f"✅ SUCCESS : '{attr_name}' (Type: {type(attr_value).__name__})")
    except Exception as e:
        # If it fails, catch the error and print it
        print(f"❌ FAILED  : '{attr_name}' (Type: {type(attr_value).__name__})")
        print(f"   -> Error: {type(e).__name__}: {e}")

print("-" * 50)
print("\nDiagnostic complete.")